<a href="https://colab.research.google.com/github/yukien462/mysite/blob/%E6%96%99%E9%87%91%E7%A2%BA%E8%AA%8D%E7%A5%A8/%E6%96%99%E9%87%91%E7%A2%BA%E8%AA%8D%E7%A5%A8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## はじめに
- 初めに以下のセルを実行してください
- 必要なライブラリのインポートとデータベース（PostgreSQL）からのデータ読み込みを行います
- pandas等、利用が想定されるライブラリは以下セルでインポートしています
- その他利用したいライブラリがあれば適宜インストールしてください（"!pip install ライブラリ名"でインストールも可能）
- 処理は複数回に分けても構いません

In [1]:
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [2]:
import os
import pandas as pd
import numpy as np

In [3]:
!pip install chardet

### PDF変換用のツール（LibreOffice）のインストール
Google Colab (Linux環境) でExcelファイルをPDFに変換するため、ヘッドレス（画面なし）で動作する LibreOffice をインストールします。  
リブレオフィス

In [4]:
# LibreOfficeのインストール（Colab環境にPDF変換エンジンを導入します）
!apt-get update
!apt-get install -y libreoffice

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:5 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:8 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libreoffice is already the newest version (4:24.2.7-0ubuntu0.24.04.7).
0 upgraded, 0 newly installed, 0 to remove and 93 not upgraded.


### 日本語フォントのインストール
LibreOfficeで日本語が文字化けするのを防ぐため、システムに日本語フォントをインストールします。

In [5]:
# 日本語フォントをインストールし、キャッシュを更新します
!apt-get install -y fonts-noto-cjk fonts-ipafont-gothic fonts-ipafont-mincho
!fc-cache -fv

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
fonts-noto-cjk is already the newest version (1:20230817+repack1-3).
fonts-ipafont-gothic is already the newest version (00303-21ubuntu1).
fonts-ipafont-mincho is already the newest version (00303-21ubuntu1).
0 upgraded, 0 newly installed, 0 to remove and 93 not upgraded.
Font directories:
	/root/.local/share/fonts
	/usr/local/share/fonts
	/usr/share/fonts
	/root/.fonts
	/usr/share/fonts/cMap
	/usr/share/fonts/cmap
	/usr/share/fonts/opentype
	/usr/share/fonts/truetype
	/usr/share/fonts/cmap/adobe-cns1
	/usr/share/fonts/cmap/adobe-gb1
	/usr/share/fonts/cmap/adobe-japan1
	/usr/share/fonts/cmap/adobe-japan2
	/usr/share/fonts/cmap/adobe-korea1
	/usr/share/fonts/opentype/ipafont-gothic
	/usr/share/fonts/opentype/ipafont-mincho
	/usr/share/fonts/opentype/linux-libertine
	/usr/share/fonts/opentype/noto
	/usr/share/fonts/truetype/crosextra
	/usr/share/fonts/truetype/dejavu
	/usr/share/fonts/truetyp

In [6]:
"""
データ読み込み
"""
import pandas as pd
from pathlib  import Path
def Read_CSV():

  Read_folder_path = Path('/content/drive/MyDrive/請求書発行/Data')
  csv_files = sorted(Read_folder_path.glob('*.csv'))

  # CSVファイルの最初の3行をテキストとして読み込む（エラーを避けるため）
  with open(csv_files[0], mode='r', encoding='SHIFT_JIS') as f:
      # 最初の3行をそれぞれ取得して、前後の空白や改行を削除したリストを作る
      head_lines = [f.readline().strip().strip("'\"") for _ in range(3)] # i in range(3) のiを使わないため"_"を表記
  #head_data = pd.DataFrame({"ヘッダー情報": head_lines})
  # 4行目以降（実データ）をデータフレームとして読み込む
  dfs = [pd.read_csv(csv_file, skiprows=3, encoding='SHIFT_JIS') for csv_file in csv_files]
  df = pd.concat(dfs, ignore_index=True)
  return df

In [7]:
"""
料金確認票用にデータを加工
"""
def Create_BillState(df):
  # 研究室ごとに
  df_by_LaboCode = df[['利用者の研究室コード','利用者の研究室','利用料金（小計）']].groupby(['利用者の研究室コード','利用者の研究室']).agg({'利用料金（小計）':'sum'}).reset_index()
  display(df_by_LaboCode)

  df_contents = df[['利用者の研究室コード','利用設備','利用者の研究室','利用者','利用料金（単価）','利用時間','利用料金（単位数）','利用料金（小計）','研究室費、私費']]
  #display(df_contents[df_contents['利用者の研究室コード'] =='BIOSF'])

  return df_contents

### PDF出力関数の実装
LibreOfficeのコマンドを使って、保存したExcelファイルをPDFに変換する関数 `Print_PDF` の実装例です。　AIさんが作成

In [8]:
"""
定数宣言
"""
# template名称
equip_dict = {
  "AFM"       : "AFM_tmplate.xlsx",
  "ECA500"    : "ECA500_template.xlsx",
  "ECX400"    : "ECX400_template.xlsx",
  "ECX400S"   : "ECX400S_template.xlsx",
  "ESI"       : "ESI_template.xlsx",
  "IT710HR"   : "IT710HR_template.xlsx",
  "JEM1400"   : "JEM1400_template.xlsx",
  "JMS700"    : "JMS700_template.xlsx",
  "MALDI"     : "MALDI_template.xlsx",
  "RAPID"     : "RAPID_template.xlsx",
  "SmartLab"  : "SmartLab_template.xlsx",
  "Synergy"   : "Synergy_template.xlsx"
}


In [9]:
"""
get_template_key
利用設備名から適応する装置キーを返す
"""
def get_equip_key(equip_name):
  match equip_name:
    case "原子間力顕微鏡":
      equip_key = "AFM"
    case "500MHz核磁気共鳴装置（日本電子・ECA500）":
      equip_key  = "ECA500"
    case "400MHz固体核磁気共鳴装置（日本電子・ECX400）":
      equip_key  = "ECX400S"
    case "400MHz核磁気共鳴装置（日本電子・ECX400）":
      equip_key  = "ECX400"
    case "ESI-TOF 質量分析装置（Bruker Daltonics・micrOTOF-QII）":
      equip_key  = "ESI"
    case "走査電子顕微鏡（低真空電子顕微鏡）（日本電子・JSM-IT710HR）":
      equip_key  = "JMS700"
    case "透過型電子顕微鏡（日本電子(株)・JEM-1400）":
      equip_key  = "JEM1400"
    case "二重収束磁場型質量分析計（日本電子・JMS-700）":
      equip_key  =  "JMS700"
    case "MALDI-TOF 質量分析装置（Bruker Daltonics・autoflex speed）":
      equip_key  = "MALDI"
    case "イメージングプレート単結晶構造解析装置　(株)リガク製　R-AXIS RAPID":
      equip_key  = "RAPID"
    case "多目的強力Ｘ線回折装置（リガク・SmartLab）":
      equip_key  = "SmartLab"
    case "高輝度高速結晶構造解析装置（リガク・XtaLAB Synergy-R DW）":
      equip_key  = "Synergy"

  return equip_key

In [49]:
import subprocess
import os
import openpyxl
from openpyxl import utils
from copy import copy


def Print_PDF(df):
    """
    データフレームを元にExcelファイルを作成し、それをPDFに変換して保存します。
    """
    # Excelテンプレートのパス
    template_dir = '/content/drive/MyDrive/請求書発行/template/'
    output_PDF_dir = '/content/drive/MyDrive/請求書発行/請求書/PDF'
    output_Excel_dir = '/content/drive/MyDrive/請求書発行/請求書/Excel'

    # 研究室コードと利用設備ごとにデータを処理する例
    grouped = df.groupby(['利用者の研究室コード','利用設備'])

    for (labo_code,equip_name), group in grouped:

        # 例: 最初の行の情報をヘッダーとして書き込む
        first_row = group.iloc[0]
        template_xlsx_name = equip_dict[get_equip_key(first_row['利用設備'])]
        template_path = os.path.join(template_dir, template_xlsx_name)
        # Excelテンプレートを開く
        wb = openpyxl.load_workbook(template_path)
        ws = wb.active

        ws['B3'] = f"{first_row['利用者の研究室']}"
        ws['B5'] = f"{first_row['利用設備']}"
        ws['I12'] = f"{group['利用料金（小計）'].sum()}"
        ws['I12'].number_format = "¥#,##0"

        # データの件数に合わせて必要な行をあらかじめ挿入し、スタイルをコピーする設計
        labo_equip_count = len(group)
        start_row = 10

        # 複数件データがある場合、テンプレートの10行目（フォーマット定義行）を下へコピー・挿入する
        if labo_equip_count > 1:
            ws.insert_rows(start_row + 1, amount=labo_equip_count - 1)
            for i in range(1, labo_equip_count):
                # 新しく挿入した行に、元の10行目のスタイルをコピー
                CopyRow(ws, src_row=start_row, dest_row=start_row + i)

        # 先ほど紹介した「一括貼り付け」関数でデータを反映させる
        detail_df = group[['利用者の研究室', '利用者', '利用料金（単価）', '利用時間','利用料金（単位数）', '利用料金（小計）']]
        Paste_DataFrame_To_Excel(ws, detail_df, start_row=start_row, start_col=1)
        detail_df = group[['研究室費、私費']]
        Paste_DataFrame_To_Excel(ws, detail_df, start_row=start_row, start_col=8)

        # 一時的なExcelファイル名を作成
        temp_xlsx_name = f"請求書_{first_row['利用者の研究室']}_{get_equip_key(first_row['利用設備'])}.xlsx"
        temp_xlsx_path = os.path.join(output_Excel_dir, temp_xlsx_name)

        # Excelとして一旦保存（元のフッター形式に戻します）
        Set_Excel_printer(ws)
        ws.oddFooter.right.text = f"{get_equip_key(first_row['利用設備'])} &P/&N "
        wb.save(temp_xlsx_path)
        wb.close()

        print(f"Excel保存完了: {temp_xlsx_name}")

        # LibreOfficeを使ってExcelをPDFに変換 (コマンドライン実行)
        cmd = [
            "libreoffice",
            "--headless",
            "--convert-to", "pdf",
            "--outdir", output_PDF_dir,
            temp_xlsx_path
        ]

        try:
            # コマンドの実行
            subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
            print(f"PDF変換完了: 請求書_{first_row['利用者の研究室']}.pdf")
        except subprocess.CalledProcessError as e:
            print(f"PDF変換エラー ({first_row['利用者の研究室']}): {e.stderr.decode('utf-8')}")


### CSV（DataFrame）のデータをExcelに一括貼り付けする関数の例
`openpyxl.utils.dataframe.dataframe_to_rows` を使うことで、ループ記述を最小限に抑えてデータを流し込むことができます。  
- データを一行ずつ取り出す  
for r_idx, row in enumerate(dataframe_to_rows(df, index=False, header=False), start=start_row):  
     r_idx : Excelの書き込み対象行 (10, 11, 12...)  
     row   : ['データA', 'データB', 'データC'] のような1行分のデータ  

In [50]:
from openpyxl.utils.dataframe import dataframe_to_rows

def Paste_DataFrame_To_Excel(ws, df, start_row=10, start_col=1):
    """
    DataFrameのデータをExcelの指定した開始行・開始列から一括で貼り付けます。
    ヘッダー(列名)やインデックスを除いて、データ部分のみを貼り付ける設定にしています。
    """
    # index=False: 行インデックスを書き込まない
    # header=False: 列ヘッダーを書き込まない（テンプレートに既に項目名がある場合）
    for r_idx, row in enumerate(dataframe_to_rows(df, index=False, header=False), start=start_row):
        for c_idx, value in enumerate(row, start=start_col):
            ws.cell(row=r_idx, column=c_idx, value=value)


In [51]:
from copy import copy

def CopyRow(ws, src_row, dest_row):
    """
    src_row から dest_row へ、値とスタイル（フォント、塗りつぶし、罫線、配置など）をコピーします。
    """
    ws.row_dimensions[dest_row].height = ws.row_dimensions[src_row].height

    # 最大列までループしてセルごとにコピー
    for col in range(1, ws.max_column + 1):
        src_cell = ws.cell(row=src_row, column=col)
        dest_cell = ws.cell(row=dest_row, column=col)

        # 値のコピー
        dest_cell.value = src_cell.value

        # スタイルのコピー (copy関数を使用することで参照ではなく複製を作成します)
        if src_cell.has_style:
            dest_cell.font = copy(src_cell.font)
            dest_cell.fill = copy(src_cell.fill)
            dest_cell.border = copy(src_cell.border)
            dest_cell.alignment = copy(src_cell.alignment)
            dest_cell.number_format = src_cell.number_format
            dest_cell.protection = copy(src_cell.protection)


----------     bottom 用紙下端からの距離  
^^^^^^^^^^     footer 用紙下端からの距離  
----------     用紙の最下端  

In [52]:
from openpyxl.utils import get_column_letter
"""
Set_Excel_printer
Excelのprinter設定
"""
def Set_Excel_printer(ws):

  # 印刷エリアの設定
  last_col = get_column_letter(ws.max_column) # エクセルの列を文字で取得
  last_row = ws.max_row
  ws.print_area = f"A1:{last_col}{last_row}"

  # 余白の設定（フッターが切れないよう安全なマージンを確保）
  ws.page_margins.top = 1.9/2.54
  ws.page_margins.bottom = 0.9/2.54  # 本文の下余白を広げる 1.5->1.3->0.9
  ws.page_margins.left = 0.5/2.54 # 0.5->0.3
  ws.page_margins.right = 0.5/2.54 # 0.5->0.3
  ws.page_margins.header = 0.2/2.54
  ws.page_margins.footer = 0.3/2.54  # フッターの印刷領域を十分に確保（bottomより小さく設定） 0.8->0.5

  # 用紙の向き　1ページの収まり設定
  ws.print_options.horizontalCentered = True # 左右（水平）中央
  ws.page_setup.orientation = ws.ORIENTATION_LANDSCAPE  # 横向き
  ws.page_setup.paperSize = ws.PAPERSIZE_A4  # A4
  ws.sheet_properties.pageSetUpPr.fitToPage = True
  ws.page_setup.fitToWidth = 1  # 横1ページに収める
  ws.page_setup.fitToHeight = 0  # 縦は自動

  # ビューをページレイアウトにすることでヘッダー・フッターの書き込みを安定化させる
  ws.views.sheetView[0].view = 'pageLayout'


In [53]:
def main():
  df = Read_CSV()           # CSV読み取り
  df = Create_BillState(df) # データ加工
  Print_PDF(df)             # PDF出力
  print("/========================  処理完了  ===========================/")

In [48]:
if __name__=="__main__":
  main()

,利用者の研究室コード,利用者の研究室,利用料金（小計）
0,BIOSF,バイオマス構造機能学研究室（堀川）,5400
1,BNK,中田研究室,25900
2,CHM,平野研究室,3700
3,FKT,加納研究室,7400
4,FSA,齊藤亜紀夫研究室,7400
5,GMT,村岡研究室,74000
6,KTA,寺田・利谷研究室,155400
7,LKR,川野研究室,18500
8,LMT,モリ研究室,25900
9,LOH,中村暢文研究室,7400


Excel保存完了: 請求書_バイオマス構造機能学研究室（堀川）_RAPID.xlsx
PDF変換完了: 請求書_バイオマス構造機能学研究室（堀川）.pdf
Excel保存完了: 請求書_中田研究室_ESI.xlsx
PDF変換完了: 請求書_中田研究室.pdf
Excel保存完了: 請求書_平野研究室_MALDI.xlsx
PDF変換完了: 請求書_平野研究室.pdf
Excel保存完了: 請求書_加納研究室_ESI.xlsx
PDF変換完了: 請求書_加納研究室.pdf
Excel保存完了: 請求書_齊藤亜紀夫研究室_ESI.xlsx
PDF変換完了: 請求書_齊藤亜紀夫研究室.pdf
Excel保存完了: 請求書_村岡研究室_MALDI.xlsx
PDF変換完了: 請求書_村岡研究室.pdf
Excel保存完了: 請求書_寺田・利谷研究室_ESI.xlsx
PDF変換完了: 請求書_寺田・利谷研究室.pdf
Excel保存完了: 請求書_川野研究室_MALDI.xlsx
PDF変換完了: 請求書_川野研究室.pdf
Excel保存完了: 請求書_モリ研究室_MALDI.xlsx
PDF変換完了: 請求書_モリ研究室.pdf
Excel保存完了: 請求書_中村暢文研究室_ESI.xlsx
PDF変換完了: 請求書_中村暢文研究室.pdf
Excel保存完了: 請求書_桜井研究室_MALDI.xlsx
PDF変換完了: 請求書_桜井研究室.pdf
Excel保存完了: 請求書_田中新垣吉野研究室_ESI.xlsx
PDF変換完了: 請求書_田中新垣吉野研究室.pdf
Excel保存完了: 請求書_栄養生理化学研究室（三浦）_ESI.xlsx
PDF変換完了: 請求書_栄養生理化学研究室（三浦）.pdf
Excel保存完了: 請求書_赤木研究室_ESI.xlsx
PDF変換完了: 請求書_赤木研究室.pdf
Excel保存完了: 請求書_吉野研究室_MALDI.xlsx
PDF変換完了: 請求書_吉野研究室.pdf
/========================  処理完了  ===========================/
